# Visualization Data Export
### Energy Consumption Prediction in France — Data Storytelling

This notebook prepares the JSON files consumed by the interactive D3.js visualizations in `docs/index.html`.  
It does not produce any charts itself — its sole purpose is to transform model outputs into the structured formats expected by each chart.

**Inputs**
- `data/energy_source_with_predictions.csv` — full dataset with Random Forest predictions  
- `data/rf_results.csv` — test-set observations with prediction errors  

**Outputs** (written to `data/`)

| File | Chart | Content |
|:-----|:------|:--------|
| `region_metrics.json` | Map tooltips | Per-region aggregated error stats |
| `rf_scatter.json` | Scatter | Observation-level predicted vs actual |
| `anova_results.json` | ANOVA chart | F-statistic, p-value, per-region MAPE |
| `violin_data.json` | Violin | Observation-level errors by region |
| `viz_e_data.json` | Density scatter | Mean MAPE and population density per region |

---

## 1. Setup

In [ ]:
import pandas as pd
import json
from scipy import stats

In [39]:
# Load full dataset with predictions
energy_df = pd.read_csv("../data/energy_source_with_predictions.csv")
# Load test results with errors
rf_results = pd.read_csv("../data/rf_results.csv")

## 2. Data Overview

Structural checks on both input datasets.  
All columns should be complete at this stage — missing value handling was done in `01_data_cleaning.ipynb`.

In [ ]:
# Schema check — full dataset with predictions
summary_energy = pd.DataFrame({
    'Unique Values':      energy_df.nunique(),
    'Data Type':          energy_df.dtypes,
    'Missing Values':     energy_df.isnull().sum(),
    'Missing Values (%)': (energy_df.isnull().sum() / len(energy_df)) * 100,
}).sort_values('Missing Values (%)', ascending=False)

print(f"energy_df  —  {energy_df.shape[0]:,} rows × {energy_df.shape[1]} columns\n")
print(summary_energy)

In [ ]:
# Error metric summary — test-set results
# These are the columns we'll use throughout the notebook.
print(f"rf_results  —  {rf_results.shape[0]:,} rows × {rf_results.shape[1]} columns\n")
print("Key error metrics (test set):")
print(rf_results[['y_true', 'y_pred_rf', 'error_rf', 'abs_error_rf', 'ape_rf']].describe().round(4))

In [ ]:
# Confirm all 12 metropolitan regions are present (Corsica excluded from the study)
print(f"Regions ({energy_df['region_name'].nunique()}):")
for r in sorted(energy_df['region_name'].unique()):
    print(f"  {r}")

### Methodological note — error metrics

The **Absolute Percentage Error (APE)** is computed at the observation level as:

$$APE = \\frac{|y_{true} - y_{pred}|}{y_{true} + \\epsilon}$$

where $\\epsilon = 10^{-8}$ is a small constant added to avoid division by zero. 
Its effect is negligible given that consumption values are in the tens of kWh per capita.

**Example:** for $y_{true} = 40.0$ and $y_{pred} = 42.0$:

| Metric | Value |
|:-------|------:|
| Error ($y_{true} - y_{pred}$) | −2.0 |
| Absolute error | 2.0 |
| APE | 0.05 → 5% |

The **regional MAPE** is the mean of all individual APEs for test observations 
belonging to that region:

$$MAPE_{region} = \\frac{1}{n} \\sum_{i=1}^{n} APE_i$$

All error metrics are computed **on the held-out test set only** (`rf_results` derives 
from `X_test` / `y_test`). This ensures we measure generalization to unseen data, 
not in-sample fit.

## 3. Export Data for Visualizations

Each section below computes the aggregated or filtered data for one chart and writes it to a JSON file in `data/`.  
Run all cells in order to regenerate the outputs.

---

### 3.1 Region-level Metrics — Map Tooltips

Per-region aggregates (MAPE, MAE, bias, density, observation count) used as tooltip data in the regional map.

In [43]:
tooltip_data = rf_results.groupby('region_name').agg(
    mean_mape      = ('ape_rf',       'mean'),
    mean_mae       = ('abs_error_rf', 'mean'),
    mean_predicted = ('y_pred_rf',    'mean'),
    mean_true      = ('y_true',       'mean'),
    mean_bias      = ('error_rf',     'mean'),
    density        = ('density',      'mean'),
    n_obs          = ('y_true',       'count')
).round(3).reset_index()

# Convert to dict keyed by region name — easy to look up in D3
metrics_dict = tooltip_data.set_index('region_name').to_dict(orient='index')

with open('../data/region_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics_dict, ensure_ascii=False, indent=2))

{
  "Auvergne-Rhône-Alpes": {
    "mean_mape": 0.04,
    "mean_mae": 1.8,
    "mean_predicted": 45.427,
    "mean_true": 45.352,
    "mean_bias": -0.076,
    "density": 112.586,
    "n_obs": 584
  },
  "Bourgogne-Franche-Comté": {
    "mean_mape": 0.051,
    "mean_mae": 2.002,
    "mean_predicted": 41.075,
    "mean_true": 40.954,
    "mean_bias": -0.121,
    "density": 58.586,
    "n_obs": 523
  },
  "Bretagne": {
    "mean_mape": 0.045,
    "mean_mae": 1.609,
    "mean_predicted": 36.954,
    "mean_true": 36.506,
    "mean_bias": -0.448,
    "density": 121.641,
    "n_obs": 569
  },
  "Centre-Val de Loire": {
    "mean_mape": 0.049,
    "mean_mae": 1.914,
    "mean_predicted": 39.756,
    "mean_true": 39.849,
    "mean_bias": 0.093,
    "density": 65.3,
    "n_obs": 324
  },
  "Grand Est": {
    "mean_mape": 0.042,
    "mean_mae": 1.797,
    "mean_predicted": 44.569,
    "mean_true": 44.519,
    "mean_bias": -0.05,
    "density": 96.29,
    "n_obs": 581
  },
  "Hauts-de-France": {
  

### 3.2 Scatter Data — Predicted vs Actual

Observation-level records for the scatter plot of $\hat{y}$ vs $y$, coloured by region.  
Only the columns needed by the chart are kept to minimise file size.

In [44]:
viz_b_data = rf_results[['y_true', 'y_pred_rf', 'abs_error_rf', 'ape_rf', 'region_name']].copy()
viz_b_data = viz_b_data.round(3)

viz_b_data.to_json('../data/rf_scatter.json', orient='records')
print(f"Total points: {len(viz_b_data)}")
print(f"y_true:  {viz_b_data['y_true'].min():.2f} — {viz_b_data['y_true'].max():.2f}")
print(f"y_pred:  {viz_b_data['y_pred_rf'].min():.2f} — {viz_b_data['y_pred_rf'].max():.2f}")
print(f"n regions: {viz_b_data['region_name'].nunique()}")

Total points: 6348
y_true:  18.96 — 74.95
y_pred:  21.11 — 71.49
n regions: 12


### 3.3 ANOVA — Are Regional Error Differences Statistically Significant?

A one-way ANOVA tests whether mean APE differs significantly across regions.

$$H_0: \mu_1 = \mu_2 = \cdots = \mu_{12} \quad \text{(all regions have the same mean APE)}$$
$$H_1: \text{at least one region differs}$$

A large F-statistic and $p \ll 0.05$ confirm that the spatial heterogeneity observed in the map is not random noise.

In [ ]:
# One-way ANOVA across regions
groups = [group['ape_rf'].values for _, group in rf_results.groupby('region_name')]
f_stat, p_value = stats.f_oneway(*groups)

# Per-region descriptive stats, sorted from worst to best MAPE
anova_table = rf_results.groupby('region_name').agg(
    n         = ('ape_rf', 'count'),
    mean_mape = ('ape_rf', 'mean'),
    std_mape  = ('ape_rf', 'std'),
    min_mape  = ('ape_rf', 'min'),
    max_mape  = ('ape_rf', 'max'),
).round(4).reset_index().sort_values('mean_mape', ascending=False)

print(f"F-statistic : {f_stat:.3f}")
print(f"p-value     : {p_value:.2e}\n")
print(anova_table.to_string(index=False))

In [46]:
anova_data = {
    "f_stat": round(f_stat, 3),
    "p_value": 2.36e-18,
    "regions": anova_table.rename(columns={
        'region_name': 'name',
        'n':           'n',
        'mean_mape':   'mean',
        'std_mape':    'std',
        'min_mape':    'min',
        'max_mape':    'max'
    }).to_dict(orient='records')
}

with open('../data/anova_results.json', 'w', encoding='utf-8') as f:
    json.dump(anova_data, f, ensure_ascii=False, indent=2)

print("Saved.")

Saved.


### 3.4 Boxplot Data — Error Distribution by Region

Full observation-level errors exported for violin/box plots.  
Regions are ordered by **median MAPE** (worst → best) so the chart reads consistently.

In [ ]:
violin_data = rf_results[['region_name', 'error_rf', 'ape_rf']].copy().round(4)

# Region order used by the chart axis (worst → best median MAPE)
region_order = (rf_results.groupby('region_name')['ape_rf']
                .median()
                .sort_values(ascending=False)
                .index.tolist())

print("Region order (worst → best median MAPE):")
for r in region_order:
    print(f"  {r}")

print(f"\nTotal rows : {len(violin_data)}")
print(f"error_rf   : {violin_data['error_rf'].min():.3f} — {violin_data['error_rf'].max():.3f}")
print(f"ape_rf     : {violin_data['ape_rf'].min():.4f} — {violin_data['ape_rf'].max():.4f}")

violin_data.to_json('../data/violin_data.json', orient='records')
print("\nSaved → data/violin_data.json")

### 3.5 Density vs MAPE Scatter

Region-level aggregates to explore whether **population density** explains prediction error.  
Each point in the chart represents one region.

In [49]:
viz_e_data = rf_results.groupby('region_name').agg(
    mean_mape = ('ape_rf', 'mean'),
    std_mape  = ('ape_rf', 'std'),
    density   = ('density', 'mean'),
).round(4).reset_index()

print(viz_e_data.sort_values('density', ascending=False).to_string(index=False))
viz_e_data.to_json('../data/viz_e_data.json', orient='records')
print("\nSaved.")


               region_name  mean_mape  std_mape   density
             Île-de-France     0.0514    0.0445 1009.7952
           Hauts-de-France     0.0408    0.0354  187.5871
Provence-Alpes-Côte d'Azur     0.0337    0.0306  159.4117
                  Bretagne     0.0448    0.0476  121.6410
          Pays de la Loire     0.0417    0.0371  115.4182
      Auvergne-Rhône-Alpes     0.0401    0.0351  112.5864
                 Normandie     0.0416    0.0347  110.5998
                 Grand Est     0.0417    0.0421   96.2902
                 Occitanie     0.0417    0.0391   79.9892
        Nouvelle-Aquitaine     0.0367    0.0343   70.1625
       Centre-Val de Loire     0.0490    0.0394   65.3002
   Bourgogne-Franche-Comté     0.0513    0.0509   58.5862

Saved.
